In [1]:
data_stream = [
    '{"id": 1, "status": "active"}',
    '{"id": 2, "status": "pending"}',
    '{"id": 3, "status": "active}',
    '{"id": 4, "status": "inactive"}',
    '{"id": 5, "status": "active"}',
    '{"id": 6, "bad json string", "hello"}'
]

In [2]:
import json
import logging

logging.basicConfig(level=logging.INFO)

In [3]:
def process_json_pipeline(raw_payload):
    logging.info("Starting pipeline")
    cleaned_records = []
    dead_letter_queue = []

    for raw_data in raw_payload:
        try:
            parsed_json = json.loads(raw_data)
            cleaned_records.append(parsed_json)
        except json.JSONDecodeError as e:
            logging.info(f"Malformed record: {e}, Raw data: {raw_data}")
            dead_letter_queue.append(raw_data)
            continue

    logging.info(f"Dead letter queue: {dead_letter_queue}")
    return cleaned_records

In [4]:
process_json_pipeline(data_stream)

INFO:root:Starting pipeline
INFO:root:Malformed record: Unterminated string starting at: line 1 column 21 (char 20), Raw data: {"id": 3, "status": "active}
INFO:root:Malformed record: Expecting ':' delimiter: line 1 column 28 (char 27), Raw data: {"id": 6, "bad json string", "hello"}
INFO:root:Dead letter queue: ['{"id": 3, "status": "active}', '{"id": 6, "bad json string", "hello"}']


[{'id': 1, 'status': 'active'},
 {'id': 2, 'status': 'pending'},
 {'id': 4, 'status': 'inactive'},
 {'id': 5, 'status': 'active'}]